# 4 · Agente

Lección 6 del módulo 1: [Agente](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239232-lesson-6-agent). El notebook original es [`agent.ipynb`](../../langchain-academy/module-1/agent.ipynb).

El router ejecuta una herramienta y termina. Un agente, en cambio, le devuelve el resultado al modelo para que decida qué hacer después: llamar otra herramienta o responder. Es el ciclo que se conoce como **ReAct**:

1. **Actuar**: el modelo pide una herramienta.
2. **Observar**: la herramienta se ejecuta y su resultado vuelve al modelo.
3. **Razonar**: con ese resultado, el modelo decide si sigue o responde.

En el grafo, el cambio respecto del router es una sola arista: de `tools` de vuelta al modelo.

## El modelo

Estos apuntes usan un modelo local con [Ollama](https://ollama.com), gratis y sin claves: `qwen2.5:7b`, que sabe llamar herramientas. Antes de ejecutar, con Ollama abierto:

```
ollama pull qwen2.5:7b
```

`temperature=0` hace que el modelo elija siempre la opción más probable, así las respuestas casi no cambian de una ejecución a otra.

El curso usa `gpt-4o` de OpenAI. Para usarlo, cambia la celda de abajo por:

```python
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o", temperature=0)
```

El resto del notebook queda igual: al grafo no le importa qué modelo hay detrás.

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen2.5:7b", temperature=0)

## Las herramientas

Tres operaciones, cada una con su docstring:

In [ ]:
def sumar(a: int, b: int) -> int:
    """Suma a y b.

    Args:
        a: primer entero
        b: segundo entero
    """
    return a + b


def multiplicar(a: int, b: int) -> int:
    """Multiplica a por b.

    Args:
        a: primer entero
        b: segundo entero
    """
    return a * b


def dividir(a: int, b: int) -> float:
    """Divide a por b.

    Args:
        a: primer entero
        b: segundo entero
    """
    return a / b


herramientas = [sumar, multiplicar, dividir]
llm_con_herramientas = llm.bind_tools(herramientas)

El curso usa `bind_tools(herramientas, parallel_tool_calls=False)`. Es una opción de la API de OpenAI: sus modelos pueden pedir varias herramientas a la vez, y aquí cada cuenta depende de la anterior, así que conviene que vayan de a una. Con este modelo no se usa.

## El nodo del modelo

El `SystemMessage` le da al modelo sus instrucciones. No se guarda en el estado: el nodo lo pone al principio de la lista en cada llamada.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import MessagesState

mensaje_sistema = SystemMessage(content="Eres un asistente que hace operaciones aritméticas con los datos que te dan.")


def asistente(estado: MessagesState):
    return {"messages": [llm_con_herramientas.invoke([mensaje_sistema] + estado["messages"])]}

## El grafo

In [ ]:
from IPython.display import Image, display
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

constructor = StateGraph(MessagesState)
constructor.add_node("asistente", asistente)
constructor.add_node("tools", ToolNode(herramientas))

constructor.add_edge(START, "asistente")
constructor.add_conditional_edges("asistente", tools_condition)
constructor.add_edge("tools", "asistente")
grafo_react = constructor.compile()

display(Image(grafo_react.get_graph(xray=True).draw_mermaid_png()))

Es el grafo del router con una diferencia: la flecha de `tools` vuelve a `asistente`. El grafo termina solo cuando el modelo responde sin pedir herramientas.

In [ ]:
resultado = grafo_react.invoke({"messages": [HumanMessage(content="Suma 3 y 4. Multiplica el resultado por 2. Divide el resultado por 5.")]})
for m in resultado["messages"]:
    m.pretty_print()

Cada `Tool Message` es una vuelta por el ciclo, y al final el modelo tiene los resultados que necesita y responde con texto. La cuenta correcta es (3 + 4) × 2 / 5 = 2,8.

## El ciclo paso a paso

Con `stream` se ve qué hizo cada nodo, en orden:

In [ ]:
for paso in grafo_react.stream({"messages": [HumanMessage(content="Suma 3 y 4. Multiplica el resultado por 2. Divide el resultado por 5.")]}, stream_mode="updates"):
    for nodo, cambio in paso.items():
        for mensaje in cambio["messages"]:
            if getattr(mensaje, "tool_calls", None):
                pedidos = ", ".join(f"{c['name']}({c['args']})" for c in mensaje.tool_calls)
                print(f"{nodo:>9} pide   {pedidos}")
            else:
                print(f"{nodo:>9} dice   {mensaje.content}")

## Un límite para el ciclo

Un ciclo podría no terminar nunca si el modelo siguiera pidiendo herramientas. Por eso LangGraph corta la ejecución después de una cantidad de pasos, `recursion_limit`, que por defecto es 25. Cada nodo que se ejecuta cuenta como un paso. Con un límite menor que los pasos que necesita esta cuenta, el grafo se detiene con un error:

In [ ]:
from langgraph.errors import GraphRecursionError

try:
    grafo_react.invoke({"messages": [HumanMessage(content="Suma 3 y 4. Multiplica el resultado por 2. Divide el resultado por 5.")]}, {"recursion_limit": 3})
except GraphRecursionError as error:
    print(type(error).__name__, "→", str(error).splitlines()[0])

## Resumen

- Un agente es un router cuya arista `tools` vuelve al modelo. El modelo decide cuándo parar.
- El ciclo es actuar, observar y razonar, y se repite hasta que el modelo responde sin pedir herramientas.
- `recursion_limit` pone un tope de pasos. Por defecto son 25.

## Ejercicios

1. Pregunta «¿Cuánto es 12 dividido por 0?». ¿Qué devuelve `ToolNode` cuando la herramienta lanza un error, y qué le responde el modelo a la persona?
2. Agrega una herramienta `potencia(a, b)` y pregunta «¿Cuánto es 2 elevado a 10, menos 24?». Mira con `stream` en qué orden llama a las herramientas.
3. Cambia el `SystemMessage` para que el modelo responda con una sola frase que incluya el resultado. ¿Cambia el número de vueltas?